# 21d - Out-of-scope leaves: what the deployed system does with crops and diseases it does not cover

Sproutly covers 11 classes in maize, tomato and potato. The models have **no "other" class**, so
inside they always score those 11. What protects a farmer who sends a bean or pepper leaf, or a
tomato disease Sproutly does not know, is the chain in front of and after the model: the blur
check, the leaf check, the crop-share guard (less than half the probability on the chosen crop
means "this does not look like ...") and the threshold (0.9557). This notebook measures how often
that chain still lets a diagnosis through. **Report only.**

| Source | What it is | Licence |
|---|---|---|
| iBean, Makerere University AI Lab, Uganda | bean leaves photographed in farmers' fields (angular leaf spot, bean rust, healthy) | MIT |
| PlantVillage, other crops | lab photos of pepper, apple, grape and other crops, in the training photos' style (the hardest case) | as notebook 01 |
| PlantDoc, other crops | web photos of other crops | CC BY 4.0 |
| PlantDoc, other tomato diseases | Septoria, mosaic virus, yellow leaf curl, leaf mould, spider mites (already on Drive) | CC BY 4.0 |

Details are in `src/data/out_of_scope.py`.

**Rules, written on 3 Nov 2026 before any photo was downloaded:**

- Scored **once**, report only. Nothing deployed (model, calibration, threshold, leaf or blur
  check) is changed because of these results before the report is handed in.
- Up to 300 photos per source, taken evenly across its folders with the project seed. You look
  at a sample sheet of every source before anything is scored.
- Each photo goes through the deployed pipeline exactly as in notebook 21c (the phone's photo,
  the server's JPEG, blur check, leaf check, ensemble, crop share at least 0.5, within-crop
  confidence at least 0.9557), sent four times: as **maize**, as **tomato**, as **potato**, and
  with **no crop** (the crop is detected).
- **Failure** = an answer: a diagnosis with advice for a leaf Sproutly does not cover. With no
  crop given, a photo that would be answered is held until the farmer confirms the guessed crop;
  it is counted separately, because a farmer who taps "Yes" then gets that answer.
- Also measured (for the question "could the model find the crop by itself?"): how sure the
  model is about the crop of these leaves, next to the same measure on notebook 21c's real
  maize and tomato photos.

A GPU runtime is best (T4); about 20-30 minutes, most of it downloads. Photos stay on Colab's disk.

In [1]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Mounted at /content/drive


In [2]:
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

Already up to date.
Refresh index: 100% (527/527), done.
## main...origin/main
 M ml/notebooks/21d_out_of_scope_leaves.ipynb
?? ml/results/fusion_quick_test/


In [3]:
%pip install -q -r {ML_DIR}/requirements.txt

## The deployed settings, and the once-only guard

In [4]:
import importlib
import io
import json
import re
import shutil

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

from src import blur_check, leaf_check, metrics, training
from src.data import d17, out_of_scope
for module in (blur_check, leaf_check, metrics, training, d17, out_of_scope):
    importlib.reload(module)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED, PER_SOURCE = config.RANDOM_SEED, 300
RESULTS = ML_DIR / "results"
LOCAL = Path("/content/out_of_scope")
OUT_DIR = config.MODELS_DIR / "out_of_scope"
REPO_RESULTS = RESULTS / "out_of_scope"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Scored once: a second run must say why (e.g. "Colab disconnected before the results were saved").
RERUN_REASON = ""
if (REPO_RESULTS / "decision.json").exists() or (OUT_DIR / "decision.json").exists():
    assert RERUN_REASON.strip(), "This test was already scored once. Write RERUN_REASON to run it again."

health = json.loads((RESULTS / "health_check" / "decision.json").read_text())
photos_b = RESULTS / "health_check_photos"
part_b = json.loads((photos_b / "decision.json").read_text())
blur = json.loads((photos_b / "blur_check.json").read_text())
leaf = json.loads((RESULTS / "leaf_check" / "leaf_check.json").read_text())

MEMBERS = health["deployed"]["model_runs"]
TEMPERATURE = health["deployed"]["temperature"]
CROP_TEMPERATURE = health["A3"]["temperature_within_crop"]
THRESHOLD = health["A3"]["threshold"]
APP_SIDE, APP_QUALITY = map(int, re.match(r"(\d+) px, q(\d+)", part_b["B1"]["app_setting"]).groups())
SERVER_QUALITY = 85
BLUR_ON, BLUR_CUT_OFF = bool(blur["switched_on"]), float(blur["cut_off"])
LEAF_CUT_OFF = float(leaf["threshold"])

CLASSES = list(config.PLANTVILLAGE_CLASSES)
CLASS_CROPS = np.array([{"Corn_(maize)": "maize", "Tomato": "tomato", "Potato": "potato"}[c.split("___")[0]]
                        for c in CLASSES])
CROPS = ["maize", "tomato", "potato"]
SHORT = {c: c.replace("Corn_(maize)___", "Maize ").replace("___", " ").replace("_", " ").strip() for c in CLASSES}
print(f"Photo {APP_SIDE} px q{APP_QUALITY} -> server q{SERVER_QUALITY} | leaf cut-off {LEAF_CUT_OFF} | "
      f"blur check {'ON, ' + str(round(BLUR_CUT_OFF, 1)) if BLUR_ON else 'off'} | T {TEMPERATURE:.4f} | "
      f"crop T {CROP_TEMPERATURE:.4f} | threshold {THRESHOLD:.4f} | {DEVICE}")

Photo 1024 px q80 -> server q85 | leaf cut-off 0.9974504172421256 | blur check ON, 22.5 | T 2.4381 | crop T 1.7002 | threshold 0.9557 | cuda


## The photos

Only the sampled files are downloaded (the Git repositories are listed without their files).
**Check the table**: every source should have up to 300 photos spread over its folders.

In [5]:
parts = [
    out_of_scope.ibean(LOCAL / "ibean", PER_SOURCE, SEED),
    out_of_scope.plantvillage_other_crops(LOCAL / "plantvillage", PER_SOURCE, SEED),
    out_of_scope.plantdoc_other_crops(LOCAL / "plantdoc", PER_SOURCE, SEED),
    out_of_scope.plantdoc_other_tomato_diseases(config.SPLITS_DIR / "unseen_test.csv", config.RAW_DATA_DIR,
                                                PER_SOURCE, SEED),
]
photos = pd.concat(parts, ignore_index=True)
pd.set_option("display.max_rows", 300)
display(photos.groupby(["source", "class"]).size().rename("photos").to_frame())
display(photos.groupby("source").size().rename("photos").to_frame())

data/train.zip: reconstructing file:   0%|          |  0.00B /  144MB            

data/train.zip: downloading bytes:           |  0.00B            

data/validation.zip: reconstructing file:   0%|          |  0.00B / 18.5MB            

data/validation.zip: downloading bytes:           |  0.00B            

data/test.zip: reconstructing file:   0%|          |  0.00B / 17.7MB            

data/test.zip: downloading bytes:           |  0.00B            

photos
source                         class                                             
ibean_bean_field               bean angular leaf spot                         100
                               bean healthy                                   100
                               bean rust                                      100
plantdoc_other_crops           Apple Scab Leaf                                 21
                               Apple leaf                                      22
                               Apple rust leaf                                 22
                               Bell_pepper leaf                                21
                               Bell_pepper leaf spot                           21
                               Blueberry leaf                                  22
                               Cherry leaf                                     21
                               Peach leaf                                      22
                               Raspberry leaf                                  22
                               Soyabean leaf                                   21
                               Squash Powdery mildew leaf                      22
                               Strawberry leaf                                 21
                               grape leaf                                      21
                               grape leaf black rot                            21
plantdoc_other_tomato_diseases Tomato Septoria leaf spot                       85
                               Tomato leaf mosaic virus                        54
                               Tomato leaf yellow virus                        75
                               Tomato mold leaf                                84
                               Tomato two spotted spider mites leaf             2
plantvillage_other_crops       Apple___Apple_scab                              14
                               Apple___Black_rot                               14
                               Apple___Cedar_apple_rust                        14
                               Apple___healthy                                 15
                               Blueberry___healthy                             15
                               Cherry_(including_sour)___Powdery_mildew        14
                               Cherry_(including_sour)___healthy               14
                               Grape___Black_rot                               14
                               Grape___Esca_(Black_Measles)                    14
                               Grape___Leaf_blight_(Isariopsis_Leaf_Spot)      14
                               Grape___healthy                                 14
                               Orange___Haunglongbing_(Citrus_greening)        15
                               Peach___Bacterial_spot                          14
                               Peach___healthy                                 14
                               Pepper,_bell___Bacterial_spot                   14
                               Pepper,_bell___healthy                          15
                               Raspberry___healthy                             14
                               Soybean___healthy                               15
                               Squash___Powdery_mildew                         15
                               Strawberry___Leaf_scorch                        14
                               Strawberry___healthy                            14

,photos
source,
ibean_bean_field,300
plantdoc_other_crops,300
plantdoc_other_tomato_diseases,300
plantvillage_other_crops,300


## STOP: look at the photos

One sheet per source, 36 random photos each. Check that each sheet shows **leaves of the crop it
says** (beans, other crops, tomato leaves with those other diseases) and **no maize, tomato or
potato leaf with one of our 11 conditions**. If a sheet looks wrong, stop and describe it to
Claude. Nothing has been scored yet.

In [6]:
REVIEW_DIR = LOCAL / "review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
sheets = []
for name, part in photos.groupby("source"):
    paths = [Path(p) for p in part.sample(min(36, len(part)), random_state=SEED)["path"]]
    sheet = d17.contact_sheet(paths, f"{name} | {len(part)} photos | {part['class'].nunique()} folders")
    target = REVIEW_DIR / f"{name}.jpg"
    sheet.save(target, quality=85)
    sheets.append(target)
    display(sheet)

Output hidden; open in https://colab.research.google.com to view.

In [8]:
# After looking at EVERY sheet above, change False to True and run this cell.
I_CHECKED_THE_PHOTOS = True

assert I_CHECKED_THE_PHOTOS, "Look at every sheet above first, then set I_CHECKED_THE_PHOTOS = True."
print("Checked. Scoring next.")

Checked. Scoring next.


## Score once, exactly as the deployed system does

Each photo is made into what the server would store (the phone's size and JPEG, then the
server's quality 85), then passed through the blur check, the leaf check and the ensemble.
The ensemble runs once per photo; the four crop choices only change which classes count.

In [ ]:
def jpeg(image, quality):
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    with Image.open(buffer) as again:
        return again.convert("RGB")


STORED = LOCAL / "stored"
STORED.mkdir(parents=True, exist_ok=True)
stored_paths, sharpness = [], []
for i, path in enumerate(tqdm(photos["path"], desc="Phone and server")):
    with Image.open(path) as original:
        image = original.convert("RGB")
    image.thumbnail((APP_SIDE, APP_SIDE), Image.BICUBIC)
    image = jpeg(jpeg(image, APP_QUALITY), SERVER_QUALITY)
    target = STORED / f"{i:05d}.jpg"
    image.save(target, format="JPEG", quality=100)
    stored_paths.append(target)
    sharpness.append(blur_check.sharpness(image))
sharpness = np.array(sharpness)

head = {"mean": np.array(leaf["feature_mean"]), "scale": np.array(leaf["feature_scale"]),
        "coef": np.array(leaf["coef"]), "intercept": leaf["intercept"]}
features, readable = leaf_check.features(leaf_check.backbone(leaf["architecture"]), stored_paths, DEVICE)
leaf_probability = leaf_check.leaf_probability(head, features)
is_leaf = readable & (leaf_probability >= LEAF_CUT_OFF)
sharp = sharpness >= BLUR_CUT_OFF if BLUR_ON else np.ones(len(photos), bool)


class Member:
    def __init__(self, path):
        saved = torch.load(path, map_location="cpu", weights_only=False)
        assert saved["classes"] == CLASSES, path
        self.network = training.build_model(len(CLASSES), saved["architecture"], pretrained=False)
        self.network.load_state_dict(saved["state_dict"])
        self.network = self.network.to(DEVICE).eval()
        size = saved["image_size"]
        self.transform = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
                                             transforms.Normalize(saved["mean"], saved["std"])])


MODELS = [Member(config.MODELS_DIR / run / "model.pt") for run in MEMBERS]
scores = np.zeros((len(photos), len(CLASSES)), np.float32)
with torch.no_grad():
    for start in tqdm(range(0, len(photos), 32), desc="Ensemble"):
        images = []
        for p in stored_paths[start:start + 32]:
            with Image.open(p) as im:
                images.append(im.convert("RGB"))
        probs = [torch.softmax(m.network(torch.stack([m.transform(im) for im in images]).to(DEVICE)).float(), dim=1)
                 for m in MODELS]
        scores[start:start + len(images)] = torch.log(torch.stack(probs).mean(dim=0) + 1e-12).cpu().numpy()
np.save(OUT_DIR / "scores.npy", scores)

Phone and server:   0%|          | 0/1200 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 84.2MB/s]


Ensemble:   0%|          | 0/38 [00:00<?, ?it/s]

## What the app would show, for each crop choice

The same order as the server: blur check, then leaf check, then the crop-share guard, then the
threshold. With no crop given, the crop with the most probability is the guess; less than half
means "Crop unclear", and a photo that would be answered waits for the farmer to confirm the crop.

In [ ]:
n = len(photos)
eleven = metrics.softmax(scores / TEMPERATURE)
crop_totals = np.stack([eleven[:, CLASS_CROPS == c].sum(axis=1) for c in CROPS], axis=1)
rows = []
for choice in CROPS + ["not given"]:
    given = choice != "not given"
    crop_index = np.full(n, CROPS.index(choice)) if given else crop_totals.argmax(axis=1)
    in_crop = CLASS_CROPS[None, :] == np.array(CROPS)[crop_index][:, None]
    share = crop_totals[np.arange(n), crop_index]
    # Other crops' classes get -1e4, so the softmax is over the chosen crop's classes only and
    # argmax is an index into all 11 classes.
    within = metrics.softmax(np.where(in_crop, scores, -1e4) / CROP_TEMPERATURE)
    confidence, pick = within.max(axis=1), within.argmax(axis=1)
    outcome = np.select(
        [~sharp, ~is_leaf, share < 0.5, confidence < THRESHOLD],
        ["too blurred", "not a leaf", "crop mismatch" if given else "crop unclear", "referred (not sure)"],
        "ANSWERED" if given else "held: asks to confirm the guessed crop",
    )
    rows.append(pd.DataFrame({
        "photo": np.arange(n), "source": photos["source"].to_numpy(), "class": photos["class"].to_numpy(),
        "crop_choice": choice, "crop_used": np.array(CROPS)[crop_index], "crop_share": share,
        "confidence": confidence, "predicted": [SHORT[CLASSES[i]] for i in pick], "outcome": outcome}))
decisions = pd.concat(rows, ignore_index=True)
print(decisions["outcome"].value_counts())

## Results

Share of photos with each outcome, by source and crop choice. **ANSWERED is the failure**: a
diagnosis with advice for a leaf Sproutly does not cover. "Held" means the app would ask the
farmer to confirm a guessed crop; tapping "Yes" would give that answer.

In [ ]:
ORDER = ["too blurred", "not a leaf", "crop mismatch", "crop unclear", "referred (not sure)",
         "held: asks to confirm the guessed crop", "ANSWERED"]
table = pd.crosstab([decisions["source"], decisions["crop_choice"]], decisions["outcome"], normalize="index")
table = table.reindex(columns=[c for c in ORDER if c in table.columns]).fillna(0)
table.insert(0, "photos", decisions.groupby(["source", "crop_choice"]).size())
display((table.drop(columns="photos") * 100).round(1).assign(photos=table["photos"]))

overall = pd.crosstab(decisions["crop_choice"], decisions["outcome"], normalize="index")
overall = overall.reindex(columns=[c for c in ORDER if c in overall.columns]).fillna(0)
print("All sources together (% of photos):")
display((overall * 100).round(1))

failures = decisions[decisions["outcome"].isin(["ANSWERED", "held: asks to confirm the guessed crop"])]
print(f"Answered or held: {len(failures)} of {len(decisions)} photo-and-choice pairs")
if len(failures):
    display(failures.groupby(["source", "crop_choice", "predicted"]).agg(
        photos=("photo", "size"), highest_confidence=("confidence", "max")).round(4))
    examples = failures.drop_duplicates("photo").head(18)
    sheet = d17.contact_sheet([stored_paths[i] for i in examples["photo"]],
                              "Answered or held (first 18 photos)")
    sheet.save(REVIEW_DIR / "answered_or_held.jpg", quality=85)
    sheets.append(REVIEW_DIR / "answered_or_held.jpg")
    display(sheet)

## Could the model find the crop by itself?

How much of the probability the model puts on its top crop. For a crop recogniser to let farmers
skip choosing the crop, real maize, tomato and potato leaves would need a high top-crop share and
other crops a low one. Notebook 21c's photos (real field maize and tomato, crop known) are the
comparison, if its saved scores are on Drive.

In [ ]:
def top_share_table(totals, groups):
    top = totals.max(axis=1)
    frame = pd.DataFrame({"group": groups, "top crop share": top})
    return frame.groupby("group")["top crop share"].agg(
        photos="size", median="median",
        **{"share >= 0.5": lambda s: (s >= 0.5).mean(), "share >= 0.9": lambda s: (s >= 0.9).mean()})


crop_view = [top_share_table(crop_totals, photos["source"].to_numpy())]
final_scores = config.MODELS_DIR / "final_field_test" / "scores.npy"
final_photos = RESULTS / "final_field_test" / "photos.csv"
if final_scores.exists() and final_photos.exists():
    ff = pd.read_csv(final_photos)
    ff_eleven = metrics.softmax(np.load(final_scores) / TEMPERATURE)
    ff_totals = np.stack([ff_eleven[:, CLASS_CROPS == c].sum(axis=1) for c in CROPS], axis=1)
    right = np.array(CROPS)[ff_totals.argmax(axis=1)] == ff["crop"].to_numpy()
    crop_view.append(top_share_table(ff_totals, ("21c real " + ff["crop"]).to_numpy()))
    print(f"21c real field photos: crop guessed right {right.mean():.1%} of {len(ff)}")
else:
    print("Notebook 21c's saved scores were not found; only the out-of-scope leaves are shown.")
crop_view = pd.concat(crop_view)
display(crop_view.round(3))
guessed = pd.crosstab(photos["source"], np.array(CROPS)[crop_totals.argmax(axis=1)], normalize="index")
print("Which crop the model thinks the out-of-scope leaves are (% of photos):")
display((guessed * 100).round(1))

## Save the results

In [ ]:
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
table.to_csv(OUT_DIR / "outcomes_by_source.csv")
overall.to_csv(OUT_DIR / "outcomes_overall.csv")
crop_view.to_csv(OUT_DIR / "crop_share.csv")
guessed.to_csv(OUT_DIR / "crop_guessed.csv")
decisions.to_csv(OUT_DIR / "decisions.csv", index=False)
photos.drop(columns=["path"]).to_csv(OUT_DIR / "photos.csv", index=False)
answered_given = decisions[(decisions["crop_choice"] != "not given") & (decisions["outcome"] == "ANSWERED")]
training.save_json({
    "scored_once_on": pd.Timestamp.now(tz="UTC").isoformat(), "rerun_reason": RERUN_REASON or None,
    "rules_written": "2026-11-03, before any photo was downloaded",
    "pipeline": {"phone": f"{APP_SIDE} px q{APP_QUALITY}", "server_quality": SERVER_QUALITY,
                 "leaf_cut_off": LEAF_CUT_OFF, "blur_check": BLUR_ON, "blur_cut_off": BLUR_CUT_OFF,
                 "temperature": TEMPERATURE, "crop_temperature": CROP_TEMPERATURE, "threshold": THRESHOLD,
                 "model_runs": MEMBERS},
    "photos": int(len(photos)), "sources": photos.groupby("source").size().to_dict(),
    "answered_with_crop_given": {"pairs": int(len(answered_given)), "of": int((decisions["crop_choice"] != "not given").sum())},
    "overall_percent": (overall * 100).round(2).reset_index().to_dict(orient="records"),
}, OUT_DIR / "decision.json")
for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json"}:
        shutil.copy(file, REPO_RESULTS / file.name)
(REPO_RESULTS / "review_sheets").mkdir(exist_ok=True)
for sheet in sheets:
    shutil.copy(sheet, REPO_RESULTS / "review_sheets" / Path(sheet).name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))
drive.flush_and_unmount()
drive.mount("/content/drive")

## Save to GitHub

Save the notebook first (Ctrl+S).

In [ ]:
from google.colab import userdata

from src import github_sync

# Drive can hide a folder made just before the remount, so make sure it is there.
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
shutil.copy(ML_DIR / "notebooks" / "21d_out_of_scope_leaves.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/out-of-scope",
    message="Out-of-scope leaves: other crops and diseases through the deployed system, scored once",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))